In [33]:
from pyspark import sql, SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import month

import matplotlib.pyplot as plt
import numpy as np

spark = (SparkSession.builder
         .master("local")
         .appName("test")
         .getOrCreate())



In [34]:
# AirBnB
calendar = spark.read.csv('./airbnb/calendar.csv', header=True)
listings = spark.read.csv('./airbnb/listings.csv', header=True)
listings_detailed = spark.read.csv('./airbnb/listings_detailed.csv', header=True)
reviews = spark.read.csv('./airbnb/reviews.csv', header=True)
reviews_detailed = spark.read.csv('./airbnb/reviews_detailed.csv', header=True)
neighbourhoods = spark.read.csv("./airbnb/neighbourhoods.csv", header=True)

# Covid
madrid_results = spark.read.csv('./covid/madrid_results.csv', header=True)

In [42]:
calendar.registerTempTable("calendar")
madrid_results.registerTempTable("madrid_results")

calendar_by_month = spark.sql("""
    SELECT 
        MONTH(c.date) AS month, 
        COUNT(c.date) AS monthly_listings, 
        m.cases
    FROM calendar AS c
    LEFT OUTER JOIN (
        SELECT 
            MONTH(date) AS date, 
            SUM(positive) AS cases 
        FROM 
            madrid_results 
        GROUP BY 
            MONTH(date)
    ) AS m on m.date = MONTH(c.date)
    GROUP BY MONTH(c.date)
    """)

calendar_by_month.show(20)



C:\Users\Kadir\AppData\Local\Programs\Python\Python39\lib\site-packages\pyspark\sql\dataframe.py:138: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn(


AnalysisException: expression 'm.cases' is neither present in the group by, nor is it an aggregate function. Add to group by or wrap in first() (or first_value) if you don't care which value you get.;
Aggregate [month(cast(date#2364 as date))], [month(cast(date#2364 as date)) AS month#2750, count(date#2364) AS monthly_listings#2751L, cases#2749]
+- Join LeftOuter, (date#2748 = month(cast(date#2364 as date)))
   :- SubqueryAlias c
   :  +- SubqueryAlias calendar
   :     +- View (`calendar`, [listing_id#2363,date#2364,available#2365,price#2366,adjusted_price#2367,minimum_nights#2368,maximum_nights#2369])
   :        +- Relation [listing_id#2363,date#2364,available#2365,price#2366,adjusted_price#2367,minimum_nights#2368,maximum_nights#2369] csv
   +- SubqueryAlias m
      +- Aggregate [month(cast(date#2673 as date))], [month(cast(date#2673 as date)) AS date#2748, sum(cast(positive#2685 as double)) AS cases#2749]
         +- SubqueryAlias madrid_results
            +- View (`madrid_result